In [132]:
import pandas as pd
import matplotlib.pyplot as plt
import pymcel as pc
import numpy as np
import spiceypy as spy

In [133]:
Obs=pd.read_csv('observaciones_astrometricas.csv')

In [134]:
Obs

,object_key,object_name,object_type,horizons_id,datetime_utc,datetime_tdb,t_jd_tdb,ra_deg,dec_deg,rho_au,sigma_ra_arcsec,sigma_dec_arcsec,sigma_rho_au,earth_x_au,earth_y_au,earth_z_au
0,oumuamua,1I/'Oumuamua,interestelar,1I,2017-10-19 23:58:50.818,A.D. 2017-Oct-20 00:00:00.0000,2458046.5,20.270964,3.103206,0.245243,0.8,0.8,0.00004,0.890823,0.445095,-0.000025
1,oumuamua,1I/'Oumuamua,interestelar,1I,2017-10-20 23:58:50.818,A.D. 2017-Oct-21 00:00:00.0000,2458047.5,15.433589,3.571600,0.272808,0.8,0.8,0.00004,0.882716,0.460360,-0.000026
2,oumuamua,1I/'Oumuamua,interestelar,1I,2017-10-21 23:58:50.818,A.D. 2017-Oct-22 00:00:00.0000,2458048.5,11.509500,3.936825,0.302073,0.8,0.8,0.00004,0.874344,0.475485,-0.000027
3,oumuamua,1I/'Oumuamua,interestelar,1I,2017-10-22 23:58:50.818,A.D. 2017-Oct-23 00:00:00.0000,2458049.5,8.296589,4.226159,0.332618,0.8,0.8,0.00004,0.865708,0.490467,-0.000027
4,oumuamua,1I/'Oumuamua,interestelar,1I,2017-10-23 23:58:50.818,A.D. 2017-Oct-24 00:00:00.0000,2458050.5,5.638788,4.459586,0.364148,0.8,0.8,0.00004,0.856812,0.505300,-0.000027
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,apophis,99942 Apophis,NEO,99942,2021-03-17 23:58:50.814,A.D. 2021-Mar-18 00:00:00.0000,2459291.5,130.478219,1.907592,0.119196,0.5,0.5,0.00002,-0.994165,0.046440,0.000002
107,apophis,99942 Apophis,NEO,99942,2021-03-18 11:58:50.814,A.D. 2021-Mar-18 12:00:00.0000,2459292.0,130.113779,2.283574,0.119702,0.5,0.5,0.00002,-0.994667,0.037811,0.000002
108,apophis,99942 Apophis,NEO,99942,2021-03-18 23:58:50.814,A.D. 2021-Mar-19 00:00:00.0000,2459292.5,129.758124,2.656119,0.120222,0.5,0.5,0.00002,-0.995094,0.029180,0.000002
109,apophis,99942 Apophis,NEO,99942,2021-03-19 11:58:50.814,A.D. 2021-Mar-19 12:00:00.0000,2459293.0,129.411211,3.025119,0.120754,0.5,0.5,0.00002,-0.995446,0.020546,0.000002


In [135]:
to_rad=np.pi/180 

In [136]:
Coor=[] #Coordenadass respecto a la Tierra, en coordenadas esféricas (ra, dec, rho)
for i in range(24):
    Coor.append([Obs['ra_deg'].iloc[i]*to_rad,Obs['dec_deg'].iloc[i]*to_rad,Obs['rho_au'].iloc[i]])

In [137]:
Tierra=[] #Coordenadas de la Tierra respecto al Sol, en coordenadas cartesianas (x,y,z)
for i in range(24):
    Tierra.append([Obs['earth_x_au'].iloc[i],Obs['earth_y_au'].iloc[i],Obs['earth_z_au'].iloc[i]])

In [138]:

# Transformar Coor (geocéntricas esféricas: ra, dec, rho) a coordenadas heliocéntricas cartesianas
Helio = []  # [x_h, y_h, z_h] en AU

for (ra, dec, rho_i), (x_t, y_t, z_t) in zip(Coor, Tierra):
    
    # Vector del objeto respecto a la Tierra (cartesiano)
    x_g = rho_i * np.cos(dec) * np.cos(ra)
    y_g = rho_i * np.cos(dec) * np.sin(ra)
    z_g = rho_i * np.sin(dec)

    # Vector heliocéntrico del objeto = Tierra(Sol->Tierra) + geocéntrico(Tierra->objeto)
    x_h = x_t + x_g
    y_h = y_t + y_g
    z_h = z_t + z_g

    Helio.append([x_h, y_h, z_h])

Helio = np.array(Helio)
Helio

array([[1.12053977, 0.52993742, 0.01325109],
       [1.14517596, 0.53281858, 0.01696891],
       [1.16964426, 0.53561574, 0.02071273],
       [1.19395063, 0.53833235, 0.02448477],
       [1.21810072, 0.54097154, 0.02828722],
       [1.24209986, 0.54353624, 0.03212229],
       [1.26595311, 0.54602912, 0.03599213],
       [1.28966526, 0.54845268, 0.03989887],
       [1.31324086, 0.55080922, 0.04384464],
       [1.33668425, 0.55310085, 0.04783149],
       [1.35999952, 0.55532957, 0.05186149],
       [1.38319059, 0.55749721, 0.05593662],
       [1.4062612 , 0.55960548, 0.06005885],
       [1.42921488, 0.56165597, 0.06423008],
       [1.45205503, 0.56365018, 0.0684522 ],
       [1.4747849 , 0.56558949, 0.07272705],
       [1.49740756, 0.5674752 , 0.07705647],
       [1.51992598, 0.56930852, 0.08144228],
       [1.542343  , 0.57109058, 0.08588633],
       [1.56466131, 0.57282242, 0.09039049],
       [1.58688353, 0.57450501, 0.09495666],
       [1.60901213, 0.57613926, 0.09958675],
       [1.

In [139]:
vel=[]
for i in range(0, 23):
    vel.append((Helio[i+1]-Helio[i]))# Velocidad heliocéntrica del objeto entre dos observaciones consecutivas (AU/día)

In [140]:
vel

[array([0.02463619, 0.00288116, 0.00371782]),
 array([0.0244683 , 0.00279717, 0.00374382]),
 array([0.02430637, 0.0027166 , 0.00377204]),
 array([0.02415009, 0.00263919, 0.00380246]),
 array([0.02399914, 0.0025647 , 0.00383507]),
 array([0.02385325, 0.00249289, 0.00386984]),
 array([0.02371215, 0.00242356, 0.00390675]),
 array([0.02357561, 0.00235653, 0.00394576]),
 array([0.02344338, 0.00229164, 0.00398686]),
 array([0.02331527, 0.00222872, 0.00402999]),
 array([0.02319107, 0.00216764, 0.00407513]),
 array([0.0230706 , 0.00210827, 0.00412222]),
 array([0.02295368, 0.00205049, 0.00417123]),
 array([0.02284015, 0.00199421, 0.00422212]),
 array([0.02272986, 0.00193931, 0.00427485]),
 array([0.02262266, 0.00188571, 0.00432942]),
 array([0.02251842, 0.00183332, 0.00438581]),
 array([0.02241701, 0.00178206, 0.00444405]),
 array([0.02231831, 0.00173184, 0.00450416]),
 array([0.02222221, 0.00168259, 0.00456617]),
 array([0.0221286 , 0.00163424, 0.00463009]),
 array([0.02203738, 0.00158673, 0.

In [157]:
h_vec=[] # Vector momento angular específico (AU^2/día)
for i in range(0, 23):
    h_vec.append(np.cross(Helio[i], vel[i]))
h_vec

[array([ 0.00193203, -0.00383951, -0.00982719]),
 array([ 0.00194731, -0.00387213, -0.00983392]),
 array([ 0.00196409, -0.00390849, -0.00984142]),
 array([ 0.00198237, -0.00394864, -0.00984971]),
 array([ 0.00200211, -0.00399263, -0.00985879]),
 array([ 0.00202332, -0.0040405 , -0.00986869]),
 array([ 0.00204597, -0.00409231, -0.00987941]),
 array([ 0.00207004, -0.00414807, -0.00989097]),
 array([ 0.00209552, -0.00420784, -0.00990336]),
 array([ 0.00212239, -0.00427163, -0.0099166 ]),
 array([ 0.00215062, -0.00433945, -0.0099307 ]),
 array([ 0.0021802 , -0.00441133, -0.00994566]),
 array([ 0.00221109, -0.00448727, -0.00996148]),
 array([ 0.00224329, -0.00456729, -0.00997816]),
 array([ 0.00227677, -0.00465141, -0.0099957 ]),
 array([ 0.00231153, -0.00473968, -0.01001412]),
 array([ 0.00234757, -0.00483216, -0.01003342]),
 array([ 0.0023849 , -0.00492894, -0.0100536 ]),
 array([ 0.00242354, -0.00503013, -0.01007469]),
 array([ 0.00246351, -0.00513583, -0.01009669]),
 array([ 0.00250483,

In [158]:
h=[]# Momento angular específico (AU^2/día)
for i in range(0, 23):
    h.append(np.linalg.norm(h_vec[i])) 
h

[np.float64(0.010726050595848814),
 np.float64(0.010746690465948035),
 np.float64(0.010769748398111281),
 np.float64(0.010795287139260878),
 np.float64(0.010823369005274034),
 np.float64(0.010854056405758997),
 np.float64(0.010887411927822237),
 np.float64(0.010923498032355693),
 np.float64(0.010962376481554261),
 np.float64(0.011004107692034326),
 np.float64(0.011048750312373115),
 np.float64(0.01109636145232356),
 np.float64(0.011146998082674353),
 np.float64(0.011200720034516675),
 np.float64(0.011257594565636847),
 np.float64(0.0113177015798259),
 np.float64(0.011381137646591327),
 np.float64(0.011448016760938751),
 np.float64(0.011518466882346983),
 np.float64(0.01159262322793436),
 np.float64(0.011670620696471852),
 np.float64(0.011752587709373813),
 np.float64(0.011838642522848785)]

In [145]:
masa_oumuamua=1e10 # kg, estimación de la masa de Oumuamua
mu_omu=pc.constantes.G * masa_oumuamua

In [ ]:
p= [] # Parámetro de la órbita (AU)
for i in range(0, 23):
    p.append(np.linalg.norm(h_vec[i])**2 / mu_omu)

p

[np.float64(0.00017237487284765249),
 np.float64(0.00017303890441079724),
 np.float64(0.00017378224017293274),
 np.float64(0.00017460741114287836),
 np.float64(0.00017551700796237304),
 np.float64(0.00017651370249973468),
 np.float64(0.00017760025543665393),
 np.float64(0.00017877951135381794),
 np.float64(0.00018005438491427414),
 np.float64(0.00018142784426514996),
 np.float64(0.00018290290137567235),
 np.float64(0.00018448262361687702),
 np.float64(0.00018617018452144152),
 np.float64(0.00018796896946739468),
 np.float64(0.0001898827373720934),
 np.float64(0.00019191580997257195),
 np.float64(0.00019407322735067104),
 np.float64(0.00019636079852379212),
 np.float64(0.00019878501014297269),
 np.float64(0.00020135282097724598),
 np.float64(0.00020407141938617854),
 np.float64(0.00020694802131538054),
 np.float64(0.00020998974691548695)]

In [155]:
e_vec=[] # Vector excentricidad específico (AU)
for i in range(0, 23):
    e_vec.append((np.cross(vel[i], h_vec[i]) / mu_omu) - (Helio[i] / np.linalg.norm(Helio[i])))

e_vec

[array([-0.90397066, -0.42713183, -0.01083984]),
 array([-0.90660472, -0.42143749, -0.01358365]),
 array([-0.90910365, -0.41592853, -0.01624894]),
 array([-0.91147713, -0.41059466, -0.01884236]),
 array([-0.9137339 , -0.40542635, -0.02137001]),
 array([-0.91588192, -0.40041471, -0.02383749]),
 array([-0.91792841, -0.39555149, -0.02624997]),
 array([-0.91987997, -0.39082899, -0.02861219]),
 array([-0.92174262, -0.38624003, -0.03092854]),
 array([-0.92352187, -0.3817779 , -0.03320307]),
 array([-0.92522276, -0.37743632, -0.03543949]),
 array([-0.92684992, -0.37320943, -0.03764128]),
 array([-0.92840761, -0.36909171, -0.0398116 ]),
 array([-0.92989974, -0.365078  , -0.04195341]),
 array([-0.93132992, -0.36116346, -0.04406944]),
 array([-0.93270148, -0.35734353, -0.04616223]),
 array([-0.93401751, -0.35361392, -0.04823416]),
 array([-0.93528084, -0.34997059, -0.05028744]),
 array([-0.93649412, -0.34640972, -0.05232419]),
 array([-0.93765979, -0.34292772, -0.05434638]),
 array([-0.93878011,

In [156]:
e=[] # Excentricidad de la órbita
for i in range(0, 23):
    e.append(np.linalg.norm(e_vec[i]))
e

[np.float64(0.9998610151975293),
 np.float64(0.9998630838107846),
 np.float64(0.9998650013878168),
 np.float64(0.9998667751002875),
 np.float64(0.9998684114506977),
 np.float64(0.9998699163261492),
 np.float64(0.9998712950552774),
 np.float64(0.9998725524671261),
 np.float64(0.9998736929494078),
 np.float64(0.999874720502167),
 np.float64(0.9998756387808893),
 np.float64(0.9998764511207184),
 np.float64(0.9998771605320116),
 np.float64(0.9998777696599712),
 np.float64(0.9998782807112696),
 np.float64(0.999878695368852),
 np.float64(0.9998790147342485),
 np.float64(0.9998792393385207),
 np.float64(0.9998793692378841),
 np.float64(0.9998794041697451),
 np.float64(0.9998793437182044),
 np.float64(0.9998791874430907),
 np.float64(0.9998789349538885)]

En todo caso, $0<e<1$, lo que corresponde a una elipse. 

In [159]:
q = [] # Perihelio (AU)
for i in range(0, 23):
    q.append(p[i] / (1 + e[i]))
q

[np.float64(8.619342621198441e-05),
 np.float64(8.65253755677452e-05),
 np.float64(8.689698557269398e-05),
 np.float64(8.730952147256024e-05),
 np.float64(8.776427836822204e-05),
 np.float64(8.826259201098353e-05),
 np.float64(8.880584259385801e-05),
 np.float64(8.939545229183134e-05),
 np.float64(9.003287835079749e-05),
 np.float64(9.071960478584057e-05),
 np.float64(9.145713754839711e-05),
 np.float64(9.224701031580931e-05),
 np.float64(9.309080987350049e-05),
 np.float64(9.399022896252007e-05),
 np.float64(9.494714713565487e-05),
 np.float64(9.59637254084431e-05),
 np.float64(9.704248403069534e-05),
 np.float64(9.818632778484182e-05),
 np.float64(9.939850032991033e-05),
 np.float64(0.0001006824814323432),
 np.float64(0.00010204186568914004),
 np.float64(0.00010348026151518191),
 np.float64(0.00010500122944708588)]

In [160]:
Q = [] # Afelio (AU)
for i in range(0, 23):
    Q.append(p[i] / (1 - e[i]))
Q

[np.float64(1.2402426005105736),
 np.float64(1.2638308544985715),
 np.float64(1.2872890866244047),
 np.float64(1.310621449291659),
 np.float64(1.3338319245329937),
 np.float64(1.3569243339650148),
 np.float64(1.3799023481142931),
 np.float64(1.4027694951989758),
 np.float64(1.4255291693536658),
 np.float64(1.4481846383751658),
 np.float64(1.4707390509962166),
 np.float64(1.4931954436951627),
 np.float64(1.5155567471116413),
 np.float64(1.5378257920507308),
 np.float64(1.560005315120446),
 np.float64(1.5820979640785788),
 np.float64(1.604106302904782),
 np.float64(1.6260328166347418),
 np.float64(1.6478799160036388),
 np.float64(1.6696499418895026),
 np.float64(1.6913451695114987),
 np.float64(1.71296781236663),
 np.float64(1.7345200258877993)]

In [162]:
a = [] # Semieje mayor (AU)
for i in range(0, 23):
    a.append(p[i] / (1 - e[i]**2))
a

[np.float64(0.6201643969683317),
 np.float64(0.6319586899369769),
 np.float64(0.643687991804898),
 np.float64(0.6553543794066218),
 np.float64(0.6669598444057383),
 np.float64(0.678506298278542),
 np.float64(0.689995576978577),
 np.float64(0.7014294453255404),
 np.float64(0.7128096011158936),
 np.float64(0.7241376789899253),
 np.float64(0.7354152540668908),
 np.float64(0.7466438453527682),
 np.float64(0.7578249189605976),
 np.float64(0.7689598911398743),
 np.float64(0.7800501311336437),
 np.float64(0.7910969639021629),
 np.float64(0.8021016726942662),
 np.float64(0.813065501481275),
 np.float64(0.8239896572520243),
 np.float64(0.8348753121855497),
 np.float64(0.8457236056884626),
 np.float64(0.8565356463140187),
 np.float64(0.8673125135585621)]